# Lesion-segmentation benchmark — LINDA · SynthStroke · OpenADS

Standalone notebook that compares three lesion-segmentation tools on a
pilot of chronic and acute strokes.

| Dataset                     | Modality      | LINDA | SynthStroke | OpenADS |
|-----------------------------|---------------|:-----:|:-----------:|:-------:|
| ds004884 — Aphasia Recovery | T1w (chronic) | ✔     | ✔           | —       |
| ISLES 2022 (Zenodo 7153326) | DWI + FLAIR (acute) | ⚠     | ✔           | ✔       |

OpenADS is DWI-based and is auto-skipped on chronic data. LINDA was
trained on chronic strokes; running it on ISLES is informative as a
transfer-fail baseline rather than a fair comparison.

**The notebook is designed to run top-to-bottom with no prior setup.** It
will:

1. Install missing Python packages.
2. Load Neurodesk modules for the segmentation tools (gracefully skipping
   any that aren't on this system).
3. Clone ds004884 via DataLad and download + unzip ISLES-2022 from
   Zenodo if either is missing.
4. For the DataLad-managed chronic set, run `datalad get` selectively
   on the sampled subjects (no full payload download). ISLES files are
   plain NIfTIs after unzip — no per-subject fetch needed.
5. Run each tool on each eligible subject, with idempotent skip-if-done
   caching.
6. Compute Dice, IoU, volume diff, HD95 + mean surface distance, and
   per-region atlas overlap. Output as CSV + HTML report.


## 1 · Config

In [11]:
# ============================================================
# Knobs you'll touch most often
# ============================================================
CONFIG = {
    # ── Subject selection ─────────────────────────────────────
    "N_CHRONIC":           2,      # subjects from ds004884
    "N_ACUTE":             2,      # subjects from ISLES 2022
    "RANDOM_SEED":         41,
    "SUBJECT_FILTER":      None,   # ["sub-XXXX", ...] to force, else None

    # ── Tool toggles ──────────────────────────────────────────
    "RUN_LINDA":           True,
    "RUN_SYNTHSTROKE":     True,
    "RUN_OPENADS":         True,

    # ── Pipeline behaviour ────────────────────────────────────
    "SKIP_IF_DONE":        True,   # idempotent runners
    "STOP_ON_TOOL_ERROR":  False,  # one tool failing won't kill the run

    # ── Setup behaviour ───────────────────────────────────────
    "AUTO_INSTALL_PIP":    True,   # `pip install` missing packages
    "AUTO_INSTALL_DATALAD": True,  # `pip install datalad` if missing
    "AUTO_CLONE_DATASETS": True,   # `datalad install` if missing
    "AUTO_FETCH_DATA":     True,   # `datalad get` for sampled subjects

    # ── Metric behaviour ──────────────────────────────────────
    "ATLAS":               "cort-maxprob-thr25-2mm",
    "HD95":                True,

    # ── Dataset identifiers ───────────────────────────────────
    # Chronic — DataLad/OpenNeuro (metadata clone, selective payload)
    "DS_CHRONIC_ID":       "ds004884",
    "DS_CHRONIC_URL":      "https://github.com/OpenNeuroDatasets/ds004884.git",
    # Acute — ISLES 2022, single Zenodo zip (1.7 GB, BIDS layout)
    "DS_ACUTE_ID":         "ISLES-2022",
    "DS_ACUTE_URL":        "https://zenodo.org/records/7153326/files/ISLES-2022.zip?download=1",
    "DS_ACUTE_MD5":        "302ee280373cdd5c190ab763d72a7a50",

    # ── Output ────────────────────────────────────────────────
    "REPORTS_SUBDIR":      "reports/benchmarks",
    "DERIV_SUBDIR":        "derivatives",
}

# ── Resolve paths ─────────────────────────────────────────────
from pathlib import Path
import os

# PROJECT_DIR = directory containing this notebook
PROJECT_DIR = Path(globals().get("PROJECT_DIR", os.getcwd())).resolve()
REPORTS_DIR = PROJECT_DIR / CONFIG["REPORTS_SUBDIR"]
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DS_CHRONIC = PROJECT_DIR / CONFIG["DS_CHRONIC_ID"]
DS_ACUTE   = PROJECT_DIR / CONFIG["DS_ACUTE_ID"]

print(f"PROJECT_DIR  {PROJECT_DIR}")
print(f"REPORTS_DIR  {REPORTS_DIR}")
print(f"DS_CHRONIC   {DS_CHRONIC}   {'exists' if DS_CHRONIC.exists() else 'NOT YET CLONED'}")
print(f"DS_ACUTE     {DS_ACUTE}     {'exists' if DS_ACUTE.exists() else 'NOT YET CLONED'}")


PROJECT_DIR  /neurodesktop-storage/aphasia-lesion-pipeline
REPORTS_DIR  /neurodesktop-storage/aphasia-lesion-pipeline/reports/benchmarks
DS_CHRONIC   /neurodesktop-storage/aphasia-lesion-pipeline/ds004884   exists
DS_ACUTE     /neurodesktop-storage/aphasia-lesion-pipeline/ISLES-2022     NOT YET CLONED


## 2 · Bootstrap — Python packages
Installs `nibabel`, `nilearn`, `scipy`, `pandas`, `datalad` if any are missing. Set `CONFIG['AUTO_INSTALL_PIP'] = False` to disable.

In [12]:
import importlib, subprocess, sys

REQUIRED = {
    "nibabel":  "nibabel",
    "nilearn":  "nilearn",
    "scipy":    "scipy",
    "pandas":   "pandas",
    "numpy":    "numpy",
}

def _import_or_pip(pkg_name: str, pip_name: str) -> bool:
    try:
        importlib.import_module(pkg_name)
        return True
    except ImportError:
        if not CONFIG["AUTO_INSTALL_PIP"]:
            print(f"  ✘ {pkg_name} missing (AUTO_INSTALL_PIP=False)")
            return False
        print(f"  installing {pip_name} …")
        cmd = [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # `--break-system-packages` is needed on Neurodesk's PEP668 envs
        try:
            subprocess.check_call(cmd + ["--break-system-packages"])
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd)
        importlib.import_module(pkg_name)
        return True

print("Python deps:")
all_ok = True
for pkg, pip in REQUIRED.items():
    ok = _import_or_pip(pkg, pip)
    print(f"  {'✔' if ok else '✘'}  {pkg}")
    all_ok &= ok

# DataLad is bigger; only install if we'll need to clone or fetch.
if CONFIG["AUTO_INSTALL_DATALAD"] and (CONFIG["AUTO_CLONE_DATASETS"] or CONFIG["AUTO_FETCH_DATA"]):
    try:
        importlib.import_module("datalad")
        print("  ✔  datalad")
    except ImportError:
        print("  installing datalad (~1 min) …")
        cmd = [sys.executable, "-m", "pip", "install", "-q", "datalad"]
        try:
            subprocess.check_call(cmd + ["--break-system-packages"])
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd)
        print("  ✔  datalad")


Python deps:
  ✔  nibabel
  ✔  nilearn
  ✔  scipy
  ✔  pandas
  ✔  numpy
  ✔  datalad


## 3 · Bootstrap — Neurodesk modules

In a Neurodesk kernel `module.load(...)` makes external tools (LINDA, SynthStroke, OpenADS, FSL, ANTs, etc.) available on `PATH`. Outside Neurodesk, `module` is undefined — the cell prints a warning and continues. Adjust the version pins to whatever `ml av <tool>` shows on your system.

In [13]:
import shutil, module

# Single batched module.load — same form as the original linda notebook.
# Edit the version pins to match `ml av <tool>` output on your Neurodesk
# system. If `module` isn't injected (running outside Neurodesk), the
# try/except prints a warning and the rest of the notebook still runs.

if "module" in globals():
    try:
        await module.load('linda/0.5.1', 'hdbet/1.0.0', 'fsl/6.0.7.22', 'ants/2.6.5', 'synthstroke/0.0.0', 'openadscpu/1.0.0', 'freesurfer/7.4.1')
        await module.list()
    except Exception as exc:
        print(f"⚠ module.load failed: {type(exc).__name__}: {exc}")
        print("  Check the pins above against `ml av <tool>` and re-run.")
else:
    print("⚠ Neurodesk `module` API not available in this kernel.")
    print("  Tools that need a Neurodesk module won't be on PATH and their")
    print("  runners will skip with a clear error.")

# CLI availability check after the load, grouped by the tool that
# provides each binary. (Module names like `openadscpu` aren't binaries —
# they put a binary like `ads` on PATH. Checking the module name itself
# would always report ✘.)
print()
print("CLIs on PATH after module loads:")
CLI_GROUPS = [
    ("LINDA",         ["linda_predict.sh"]),
    ("HD-BET",        ["hd-bet"]),
    ("FSL",           ["fslmaths", "mri_synthstrip"]),
    ("Freesurfer",    ["mri_synthstrip"]),
    ("ANTs",          ["antsRegistrationSyN.sh", "antsApplyTransforms"]),
    ("SynthStroke",   ["synthstroke"]),
    ("OpenADS",       ["ads"]),           # module is openadscpu, binary is `ads`
    ("DataLad/annex", ["datalad", "git-annex"]),
    ("Container",     ["singularity", "apptainer"]),
]
for group, names in CLI_GROUPS:
    found = [(n, shutil.which(n)) for n in names]
    any_ok = any(p for _, p in found)
    print(f"  {'✔' if any_ok else '✘'}  {group}")
    for n, p in found:
        print(f"      {'✔' if p else '✘'} {n:25s} {p or ''}")



CLIs on PATH after module loads:
  ✔  LINDA
      ✔ linda_predict.sh          /neurodesktop-storage/containers/linda_0.5.1_20260502/linda_predict.sh
  ✔  HD-BET
      ✔ hd-bet                    /cvmfs/neurodesk.ardc.edu.au/containers/hdbet_1.0.0_20210318/hd-bet
  ✔  FSL
      ✔ fslmaths                  /cvmfs/neurodesk.ardc.edu.au/containers/fsl_6.0.7.22_20260416/fslmaths
      ✔ mri_synthstrip            /cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/mri_synthstrip
  ✔  Freesurfer
      ✔ mri_synthstrip            /cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/mri_synthstrip
  ✔  ANTs
      ✔ antsRegistrationSyN.sh    /cvmfs/neurodesk.ardc.edu.au/containers/ants_2.6.5_20260225/antsRegistrationSyN.sh
      ✔ antsApplyTransforms       /cvmfs/neurodesk.ardc.edu.au/containers/ants_2.6.5_20260225/antsApplyTransforms
  ✔  SynthStroke
      ✔ synthstroke               /neurodesktop-storage/containers/synthstroke_0.0.0_20260518/synthstroke
  ✔  OpenADS

## 4 · Bootstrap — Install datasets if missing

Two different mechanisms because the datasets live in different places:

- **Chronic — ds004884 (DataLad/OpenNeuro).** `datalad install` is metadata-only — clones the dataset structure and lesion pointer files but doesn't download the multi-gigabyte NIfTI payload. We fetch payload selectively in step 6 once we know which subjects we want.
- **Acute — ISLES 2022 (Zenodo).** Single ~1.7 GB zip with the BIDS-style dataset baked in. The cell downloads it once (md5-verified), extracts it into `PROJECT_DIR/ISLES-2022/`, then skips on subsequent runs.

In [14]:
import hashlib, urllib.request, zipfile

def datalad_install(url: str, target: Path) -> bool:
    """Clone an OpenNeuro DataLad dataset (metadata only)."""
    if target.exists():
        print(f"  ✔  {target.name} already cloned")
        return True
    if not CONFIG["AUTO_CLONE_DATASETS"]:
        print(f"  ✘  {target.name} missing (AUTO_CLONE_DATASETS=False)")
        return False
    if not shutil.which("datalad"):
        print(f"  ✘  datalad CLI missing — can't clone {target.name}")
        return False
    print(f"  cloning {target.name} from {url} …")
    proc = subprocess.run(["datalad", "install", "-s", url, str(target)],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  ✘  datalad install failed: {proc.stderr.strip()[:300]}")
        return False
    print(f"  ✔  {target.name} cloned")
    return True


def _md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


def _looks_extracted(target: Path) -> bool:
    """ISLES is considered "installed" if at least one sub-strokecase
    directory exists under target/ (or target/ISLES-2022/, if the zip
    nested itself one level deep)."""
    if not target.exists():
        return False
    for root in (target, target / target.name):
        if any(root.glob("sub-strokecase*")):
            return True
    return False


def zenodo_install(url: str, target: Path, *, md5: str | None = None) -> bool:
    """Download + extract a Zenodo zip. Idempotent: returns True if the
    extracted dataset already exists, or if download+unzip succeeded."""
    if _looks_extracted(target):
        print(f"  ✔  {target.name} already extracted")
        return True
    if not CONFIG["AUTO_CLONE_DATASETS"]:
        print(f"  ✘  {target.name} missing (AUTO_CLONE_DATASETS=False)")
        return False

    target.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target.parent / f"{target.name}.zip"

    # Download if missing (or if md5 mismatches an existing partial file)
    need_dl = True
    if zip_path.exists() and md5:
        print(f"  found existing {zip_path.name}, checking md5 …")
        if _md5(zip_path) == md5:
            need_dl = False
            print(f"  ✔  md5 matches, skipping download")
        else:
            print(f"  ✘  md5 mismatch, re-downloading")
            zip_path.unlink()

    if need_dl:
        print(f"  downloading {target.name} from Zenodo (~1.7 GB) — "
              f"this can take several minutes …")
        try:
            urllib.request.urlretrieve(url, zip_path)
        except Exception as exc:
            print(f"  ✘  download failed: {type(exc).__name__}: {exc}")
            return False
        if md5 and _md5(zip_path) != md5:
            print(f"  ✘  md5 mismatch after download — file may be corrupt")
            return False
        print(f"  ✔  download complete ({zip_path.stat().st_size:,} bytes)")

    print(f"  extracting → {target} …")
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target.parent)
    except Exception as exc:
        print(f"  ✘  extract failed: {type(exc).__name__}: {exc}")
        return False

    # Some Zenodo zips wrap content in a top-level folder of the same name.
    # If the expected sub-strokecase dirs are one level deep, lift them up.
    if not list(target.glob("sub-strokecase*")):
        nested = target / target.name
        if nested.exists() and list(nested.glob("sub-strokecase*")):
            for child in nested.iterdir():
                child.rename(target / child.name)
            nested.rmdir()

    print(f"  ✔  {target.name} ready "
          f"({len(list(target.glob('sub-strokecase*')))} subjects)")
    return True


print("Datasets:")
datalad_install(CONFIG["DS_CHRONIC_URL"], DS_CHRONIC)
zenodo_install(CONFIG["DS_ACUTE_URL"],    DS_ACUTE, md5=CONFIG["DS_ACUTE_MD5"])


Datasets:
  ✔  ds004884 already cloned
  downloading ISLES-2022 from Zenodo (~1.7 GB) — this can take several minutes …
  ✔  download complete (1,692,717,470 bytes)
  extracting → /neurodesktop-storage/aphasia-lesion-pipeline/ISLES-2022 …
  ✔  ISLES-2022 ready (250 subjects)


True

## 5 · Helpers — gzip check, annex fetch, metrics

In [ ]:
import json, glob, time
import numpy as np, pandas as pd, nibabel as nib
from nilearn.image import resample_to_img
from IPython.display import display, HTML

GZIP_MAGIC = b"\x1f\x8b"

def is_gzip(path: Path) -> bool:
    try:
        with open(path, "rb") as fh:
            return fh.read(2) == GZIP_MAGIC
    except OSError:
        return False


def datalad_get(target: Path, timeout=900, stream: bool = True) -> Path | None:
    """Materialise a single git-annex-managed file via `datalad get`.
    Falls back to `git annex get`. If `stream=True`, the command's stdout
    and stderr are printed live (so you can see download progress and
    real error messages). Returns target if fetched, else None."""
    if target is None or not target.exists():
        return None
    if is_gzip(target):  # already real content → nothing to do
        return target
    repo = target
    while repo != repo.parent and not (repo / ".git").exists():
        repo = repo.parent
    if not (repo / ".git").exists():
        print(f"  [fetch] {target.name}: no .git found above the file")
        return None
    rel = target.relative_to(repo)
    for cmd in (["datalad", "get", str(rel)],
                ["git", "annex", "get", str(rel)]):
        if not shutil.which(cmd[0]):
            continue
        print(f"  [fetch] {cmd[0]} get {rel} (cwd={repo}) …")
        try:
            if stream:
                # Inherit the parent's stdout/stderr so the user sees progress
                proc = subprocess.run(cmd, cwd=str(repo),
                                      timeout=timeout)
                rc, err = proc.returncode, ""
            else:
                proc = subprocess.run(cmd, cwd=str(repo),
                                      capture_output=True, text=True,
                                      timeout=timeout)
                rc, err = proc.returncode, (proc.stderr or proc.stdout or "")
        except subprocess.TimeoutExpired:
            print(f"  [fetch] {rel}: TIMED OUT after {timeout}s")
            return None
        if rc == 0 and is_gzip(target):
            print(f"  [fetch] {rel}: ✔ materialised "
                  f"({target.stat().st_size:,} bytes)")
            return target
        if err and not stream:
            print(f"  [fetch] {rel}: {cmd[0]} rc={rc}  err={err.strip()[:400]}")
        else:
            print(f"  [fetch] {rel}: {cmd[0]} rc={rc}")
    print(f"  [fetch] {rel}: ✘ NOT materialised — tried datalad and git-annex")
    return None


def binarise(img):
    return (img.get_fdata() > 0).astype(np.uint8)


def voxel_volume_mm3(img):
    return float(np.prod(img.header.get_zooms()[:3]))


def volume_ml(mask, vv):
    return int(mask.sum()) * vv / 1000.0


def dice_iou(a, b):
    a = a.astype(bool); b = b.astype(bool)
    inter = int((a & b).sum()); s = int(a.sum()) + int(b.sum())
    union = s - inter
    return ((2 * inter / s) if s else float("nan"),
            (inter / union) if union else float("nan"))


def surface_distances(a, b, zooms):
    """HD95 and mean surface distance in mm."""
    out = {"hd95_mm": float("nan"), "mean_surf_mm": float("nan")}
    if a.sum() == 0 or b.sum() == 0:
        return out
    try:
        from scipy.ndimage import binary_erosion, distance_transform_edt
    except ImportError:
        return out
    def boundary(m):
        return m & ~binary_erosion(m, iterations=1)
    ba = boundary(a.astype(bool)); bb = boundary(b.astype(bool))
    if ba.sum() == 0 or bb.sum() == 0:
        return out
    dt_b = distance_transform_edt(~bb, sampling=zooms)
    dt_a = distance_transform_edt(~ba, sampling=zooms)
    d = np.concatenate([dt_b[ba], dt_a[bb]])
    out["hd95_mm"]      = float(np.percentile(d, 95))
    out["mean_surf_mm"] = float(d.mean())
    return out

print("Helpers loaded ✔")


## 6 · Discover subjects and fetch their data

Walks each dataset, samples `N` subjects per dataset (seed-stable), then runs `datalad get` on the T1w / DWI / lesion-mask files for just those subjects — keeping the download proportional to the pilot size, not the full dataset.

In [ ]:
import random

def _glob_first(base: Path, *patterns) -> Path | None:
    if not base.exists():
        return None
    for pat in patterns:
        hits = sorted(base.glob(pat))
        if hits:
            return hits[0]
    return None


def _t1w(ds: Path, sub: str, ses: str | None) -> Path | None:
    base = (ds / sub / ses / "anat") if ses else (ds / sub / "anat")
    return _glob_first(base, "*T1w.nii*", "*T1w*.nii*")


def _dwi(ds: Path, sub: str, ses: str | None) -> Path | None:
    base = (ds / sub / ses / "dwi") if ses else (ds / sub / "dwi")
    return _glob_first(base, "*dwi.nii*", "*DWI*.nii*")


def discover(ds_root: Path, *, need_t1w=False, need_dwi=False) -> list[dict]:
    out = []
    if not ds_root.exists():
        return out
    for sub_dir in sorted(p for p in ds_root.iterdir()
                          if p.is_dir() and p.name.startswith("sub-")):
        sub = sub_dir.name
        sessions = [p.name for p in sorted(sub_dir.iterdir())
                    if p.is_dir() and p.name.startswith("ses-")] or [None]
        for ses in sessions:
            e = {"subject": sub, "session": ses,
                 "t1w": _t1w(ds_root, sub, ses),
                 "dwi": _dwi(ds_root, sub, ses)}
            if need_t1w and not e["t1w"]: continue
            if need_dwi and not (e["dwi"] or e["t1w"]): continue
            out.append(e)
    return out


def sample(entries: list[dict], n: int) -> list[dict]:
    if not entries:
        return []
    if CONFIG["SUBJECT_FILTER"]:
        entries = [e for e in entries if e["subject"] in CONFIG["SUBJECT_FILTER"]]
    if n is None or n >= len(entries):
        return entries
    rng = random.Random(CONFIG["RANDOM_SEED"])
    pick = rng.sample(entries, n)
    pick.sort(key=lambda e: (e["subject"], e["session"] or ""))
    return pick


SUBJECTS_CHRONIC = sample(discover(DS_CHRONIC, need_t1w=True), CONFIG["N_CHRONIC"])
SUBJECTS_ACUTE   = sample(discover(DS_ACUTE,   need_dwi=True), CONFIG["N_ACUTE"])


def _all_lesion_pointers(ds_root: Path, sub: str, ses: str) -> list[Path]:
    """Find every existing lesion-mask path for this subject across the
    BIDS derivatives layouts the supported datasets use:

    - ds004884:   derivatives/lesion_masks/<sub>/<ses>/anat/*desc-lesion_mask*
    - ISLES 2022: derivatives/<sub>/<ses>/<sub>_<ses>_msk.nii.gz
                  (a single per-subject mask, not nested under anat/)
    """
    candidates: list[Path] = []

    # ds004884-style trees and other common variants
    masked_roots = [
        ds_root / "derivatives" / "lesion_masks" / sub,
        ds_root / "derivatives" / "lesion_masks_native" / sub,
        ds_root / "derivatives" / "lesion_masks_mni" / sub,
        ds_root / "derivatives" / "manual_lesion" / sub,
    ]
    for root in masked_roots:
        if not root.exists():
            continue
        for pattern in ([f"{ses}/anat/*lesion*.nii*", f"{ses}/*lesion*.nii*"]
                        if ses else []) + ["ses-*/anat/*lesion*.nii*",
                                            "ses-*/*lesion*.nii*",
                                            "anat/*lesion*.nii*",
                                            "*lesion*.nii*"]:
            candidates.extend(root.glob(pattern))

    # ISLES 2022 layout: derivatives/<sub>/<ses>/<sub>_<ses>_msk.nii.gz
    # (no anat/ subdir, no "lesion" substring in the filename)
    isles_root = ds_root / "derivatives" / sub
    if isles_root.exists():
        for pattern in ([f"{ses}/*msk*.nii*"] if ses else []) + [
                "ses-*/*msk*.nii*", "*msk*.nii*"]:
            candidates.extend(isles_root.glob(pattern))

    return sorted(set(candidates))


def fetch_for(entry: dict, ds_root: Path) -> dict:
    """Pre-fetch payload for one subject: T1w, DWI, every expert mask.
    Returns a dict of {label: True/False/None} so the caller can summarise."""
    out = {"t1w": None, "dwi": None, "mask": None, "mask_paths": []}
    if not CONFIG["AUTO_FETCH_DATA"]:
        return out
    sub, ses = entry["subject"], entry["session"] or ""

    for key in ("t1w", "dwi"):
        p = entry.get(key)
        if p is None:
            continue
        got = datalad_get(p)
        out[key] = bool(got)

    masks = _all_lesion_pointers(ds_root, sub, ses)
    out["mask_paths"] = [str(p) for p in masks]
    if not masks:
        print(f"  [fetch] {sub}/{ses}: no lesion mask file found under "
              f"derivatives/lesion_mask* — discoverable layouts:")
        for guess in (ds_root / "derivatives").glob("lesion_*"):
            print(f"      • {guess.relative_to(ds_root)}")
        out["mask"] = False
    else:
        ok = []
        for p in masks:
            got = datalad_get(p)
            ok.append(bool(got))
        out["mask"] = any(ok)
    return out


def _bids_layout_hint(ds_root: Path, n: int = 3) -> str:
    """Tiny diagnostic: list a few subjects and their immediate children
    so the user can confirm the BIDS layout they actually have on disk."""
    lines = []
    if not ds_root.exists():
        return f"  (dataset root does not exist: {ds_root})"
    subs = sorted(p for p in ds_root.iterdir()
                  if p.is_dir() and p.name.startswith("sub-"))[:n]
    if not subs:
        return f"  (no sub-* directories under {ds_root})"
    for s in subs:
        children = sorted(c.name for c in s.iterdir())[:6]
        lines.append(f"  {s.name}/  →  {', '.join(children)}")
    if len(subs) == n:
        lines.append(f"  … and more")
    return "\n".join(lines)


# ── Discover ───────────────────────────────────────────────────────────
print(f"=== Chronic (ds004884) — discovering subjects ===")
print(_bids_layout_hint(DS_CHRONIC))
_all_chronic = discover(DS_CHRONIC, need_t1w=True)
SUBJECTS_CHRONIC = sample(_all_chronic, CONFIG["N_CHRONIC"])
print(f"  → {len(_all_chronic)} subjects with T1w, sampling {len(SUBJECTS_CHRONIC)}")

print()
print(f"=== Acute (ISLES 2022) — discovering subjects ===")
print(_bids_layout_hint(DS_ACUTE))
_all_acute = discover(DS_ACUTE, need_dwi=True)
SUBJECTS_ACUTE = sample(_all_acute, CONFIG["N_ACUTE"])
print(f"  → {len(_all_acute)} subjects with DWI/T1w, sampling {len(SUBJECTS_ACUTE)}")

if not SUBJECTS_ACUTE and DS_ACUTE.exists():
    print()
    print("⚠ ISLES 2022 is on disk but no DWI-bearing subjects matched.")
    print("  Inspect the layout above — ISLES uses `sub-strokecase####` ")
    print("  with `<sub>/<ses>/dwi/<sub>_<ses>_dwi.nii.gz`. If your")
    print("  extract sits one directory deeper (e.g. ISLES-2022/ISLES-2022/),")
    print("  re-run the install cell — it tries to lift nested content.")

# ── Fetch ──────────────────────────────────────────────────────────────
fetch_summary = []
if CONFIG["AUTO_FETCH_DATA"]:
    for label, entries, ds_root in [
        ("chronic_ds004884", SUBJECTS_CHRONIC, DS_CHRONIC),
        ("acute_isles2022",  SUBJECTS_ACUTE,   DS_ACUTE),
    ]:
        if not entries:
            continue
        print(f"\n=== Fetching payload for {len(entries)} {label} subject(s) ===")
        for e in entries:
            sub, ses = e["subject"], e["session"] or ""
            print(f"\n• {sub} {ses}")
            res = fetch_for(e, ds_root)
            fetch_summary.append({"dataset": label, "subject": sub,
                                   "session": ses, **{k: v for k, v in res.items() if k != "mask_paths"},
                                   "n_masks": len(res["mask_paths"])})

if fetch_summary:
    print()
    print("=== Fetch summary ===")
    fs_df = pd.DataFrame(fetch_summary)
    display(fs_df)
    # ISLES files are plain (post-unzip) NIfTIs — no annex fetch step.
    # But still surface a clear summary so missing layouts get noticed.
    acute_rows = fs_df[fs_df["dataset"] == "acute_isles2022"]
    if not acute_rows.empty:
        n_dwi = int((acute_rows["dwi"] == True).sum())
        n_mask = int((acute_rows["mask"] == True).sum())
        print(f"Acute (ISLES): {n_dwi}/{len(acute_rows)} subjects have DWI "
              f"on disk, {n_mask}/{len(acute_rows)} have a lesion mask on disk.")
        if n_dwi == 0 or n_mask == 0:
            print("⚠ ISLES discovery incomplete. Common causes:")
            print("    1. The Zenodo zip didn't finish extracting — re-run the")
            print("       dataset-install cell; zenodo_install() is idempotent.")
            print("    2. The extract sits one directory deeper than expected")
            print("       (ISLES-2022/ISLES-2022/sub-strokecase…/). The install")
            print("       cell tries to lift nested content, but check manually.")
            print("    3. Disk full or zip corruption — verify md5 matches")
            print(f"       {CONFIG['DS_ACUTE_MD5']}")


## 7 · Reference mask resolver

In [ ]:
def reference_mask(entry: dict, dataset_root: Path) -> tuple[Path | None, str]:
    """Locate the dataset's expert lesion mask, auto-fetching annex
    pointers. Tries multiple derivatives layouts (ds004884 vs ISLES)
    and falls back to a different session of the same subject."""
    sub, ses = entry["subject"], entry["session"] or ""

    def _try(p: Path | None, space: str):
        if p is None: return None, "missing"
        got = datalad_get(p) if not is_gzip(p) else p
        return (got, space) if got else (None, "fetch-failed")

    cands = _all_lesion_pointers(dataset_root, sub, ses)
    if not cands:
        return None, "missing"
    # Prefer native-space mask for the active session; fall back to anything
    same_session = [p for p in cands if (not ses) or ses in p.parts]
    pick = same_session[0] if same_session else cands[0]
    if same_session:
        space = "native"
    else:
        # find which session this fallback came from for the label
        ses_parts = [part for part in pick.parts if part.startswith("ses-")]
        space = f"native (alt {ses_parts[0]})" if ses_parts else "native (alt session)"
    return _try(pick, space)


## 8 · Tool runners

Each runner is idempotent (skip-if-done), prints actionable errors if its CLI isn't on PATH, and returns the prediction path or `None`.

In [ ]:
def run_linda(entry: dict, dataset_root: Path) -> Path | None:
    """LINDA on T1w → derivatives/linda/<sub>/<ses>/anat/Prediction3_native.nii.gz"""
    sub, ses = entry["subject"], entry["session"] or ""
    t1w      = entry.get("t1w")
    if not t1w:
        print(f"  [linda] {sub}/{ses}: no T1w → skip")
        return None
    out_dir = dataset_root / "derivatives" / "linda" / sub / ses / "anat"
    out_dir.mkdir(parents=True, exist_ok=True)
    pred = out_dir / "Prediction3_native.nii.gz"
    if CONFIG["SKIP_IF_DONE"] and pred.exists() and is_gzip(pred):
        print(f"  [linda] {sub}/{ses}: cached ✔")
        return pred

    wrapper = PROJECT_DIR / "linda_predict_with_mask.sh"
    if wrapper.exists():
        cmd = [str(wrapper), "--t1", str(t1w), "--outdir", str(out_dir)]
    elif shutil.which("linda_predict.sh"):
        cmd = ["linda_predict.sh", "--t1", str(t1w), "--outdir", str(out_dir)]
    else:
        print(f"  [linda] {sub}/{ses}: neither in-project wrapper nor "
              f"linda_predict.sh on PATH (did you `module.load('linda/...')`?)")
        return None

    print(f"  [linda] {sub}/{ses}: running (10–30 min) …")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  [linda] {sub}/{ses}: FAILED in {time.time()-t0:.0f}s "
              f"rc={proc.returncode}\n    {proc.stderr.strip()[:400]}")
        return None
    print(f"  [linda] {sub}/{ses}: done in {time.time()-t0:.0f}s")
    return pred if pred.exists() else None


In [ ]:
def _synthstroke_cmd() -> list[str] | None:
    """Auto-detect how SynthStroke is exposed in this environment."""
    for binname in ("synthstroke", "SynthStrokeSPM"):
        if shutil.which(binname):
            return [binname]
    # Python module fallback (the upstream repo's typical entrypoint)
    try:
        r = subprocess.run([sys.executable, "-c", "import synthstroke"],
                           capture_output=True)
        if r.returncode == 0:
            return [sys.executable, "-m", "synthstroke"]
    except Exception:
        pass
    return None


def run_synthstroke(entry: dict, dataset_root: Path) -> Path | None:
    sub, ses = entry["subject"], entry["session"] or ""
    inp = entry.get("t1w") or entry.get("dwi")  # SynthStroke is cross-sequence
    if not inp:
        print(f"  [synthstroke] {sub}/{ses}: no T1w/DWI → skip")
        return None
    out_dir = dataset_root / "derivatives" / "synthstroke" / sub / ses / "anat"
    out_dir.mkdir(parents=True, exist_ok=True)
    pred = out_dir / f"{sub}_{ses}_desc-synthstroke_lesion.nii.gz".replace("__", "_")
    if CONFIG["SKIP_IF_DONE"] and pred.exists() and is_gzip(pred):
        print(f"  [synthstroke] {sub}/{ses}: cached ✔")
        return pred

    base_cmd = _synthstroke_cmd()
    if base_cmd is None:
        print(f"  [synthstroke] {sub}/{ses}: not on PATH "
              f"(did you `module.load('synthstroke/<ver>')`?)")
        return None

    # Typical CLI: synthstroke --input <in> --output <mask>
    # Adjust the flags if your build differs.
    cmd = base_cmd + ["--input", str(inp), "--output", str(pred)]
    print(f"  [synthstroke] {sub}/{ses}: running …")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  [synthstroke] {sub}/{ses}: FAILED in {time.time()-t0:.0f}s "
              f"rc={proc.returncode}\n    {proc.stderr.strip()[:400]}")
        return None
    print(f"  [synthstroke] {sub}/{ses}: done in {time.time()-t0:.0f}s")
    return pred if pred.exists() else None


In [ ]:
def _openads_cmd() -> list[str] | None:
    for binname in ("ads", "openads"):
        if shutil.which(binname):
            return [binname]
    try:
        r = subprocess.run([sys.executable, "-c", "import ads"],
                           capture_output=True)
        if r.returncode == 0:
            return [sys.executable, "-m", "ads.cli"]
    except Exception:
        pass
    return None


def run_openads(entry: dict, dataset_root: Path) -> Path | None:
    """OpenADS DWI pipeline → derivatives/openads/<sub>/DWI/postprocessing/*lesion*.nii.gz"""
    sub, ses = entry["subject"], entry["session"] or ""
    if not entry.get("dwi"):
        print(f"  [openads] {sub}/{ses}: no DWI → skip (OpenADS needs DWI)")
        return None
    subject_folder = (dataset_root / sub / ses) if ses else (dataset_root / sub)
    out_root       = dataset_root / "derivatives" / "openads"
    out_root.mkdir(parents=True, exist_ok=True)

    expected = out_root / sub / "DWI" / "postprocessing" / "lesion_native.nii.gz"
    if CONFIG["SKIP_IF_DONE"] and expected.exists() and is_gzip(expected):
        print(f"  [openads] {sub}/{ses}: cached ✔")
        return expected

    base_cmd = _openads_cmd()
    if base_cmd is None:
        print(f"  [openads] {sub}/{ses}: `ads` CLI not on PATH "
              f"(did you `module.load('openadscpu/<ver>')`?)")
        return None

    cmd = base_cmd + ["dwi",
                      "--subject-path", str(subject_folder),
                      "--output-root",  str(out_root),
                      "--all"]
    print(f"  [openads] {sub}/{ses}: running DWI pipeline …")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"  [openads] {sub}/{ses}: FAILED in {time.time()-t0:.0f}s "
              f"rc={proc.returncode}\n    {proc.stderr.strip()[:400]}")
        return None
    print(f"  [openads] {sub}/{ses}: done in {time.time()-t0:.0f}s")
    if expected.exists():
        return expected
    cands = sorted((out_root / sub / "DWI" / "postprocessing")
                   .rglob("*lesion*.nii.gz")) if (out_root / sub).exists() else []
    return cands[-1] if cands else None


## 9 · Orchestrator — run every tool on every subject

In [ ]:
RUNNERS = {
    "linda":       (CONFIG["RUN_LINDA"],       run_linda),
    "synthstroke": (CONFIG["RUN_SYNTHSTROKE"], run_synthstroke),
    "openads":     (CONFIG["RUN_OPENADS"],     run_openads),
}


def run_dataset(label, entries, ds_root, eligible):
    rows = []
    print(f"\n=== {label}: {len(entries)} subject(s) ===")
    for e in entries:
        sub, ses = e["subject"], e["session"] or ""
        print(f"\n• {sub} {ses}")
        ref_path, ref_space = reference_mask(e, ds_root)
        if ref_path is None:
            print(f"  [ref] no expert mask ({ref_space}); metrics will be NaN")
        else:
            print(f"  [ref] {ref_path.name}  ({ref_space})")

        row = {"dataset": label, "subject": sub, "session": ses,
               "reference": str(ref_path) if ref_path else "",
               "reference_space": ref_space}
        for tool, (enabled, fn) in RUNNERS.items():
            if not enabled or tool not in eligible:
                row[f"{tool}_pred"] = ""
                continue
            try:
                pred = fn(e, ds_root)
                row[f"{tool}_pred"] = str(pred) if pred else ""
            except Exception as exc:
                print(f"  [{tool}] {sub}/{ses}: EXCEPTION "
                      f"{type(exc).__name__}: {exc}")
                row[f"{tool}_pred"] = ""
                if CONFIG["STOP_ON_TOOL_ERROR"]:
                    raise
        rows.append(row)
    return rows


_chronic_eligible = {"linda", "synthstroke"}            # no DWI
_acute_eligible   = {"linda", "synthstroke", "openads"} # LINDA is off-label

RUN_LOG = []
RUN_LOG += run_dataset("chronic_ds004884", SUBJECTS_CHRONIC, DS_CHRONIC, _chronic_eligible)
RUN_LOG += run_dataset("acute_isles2022",  SUBJECTS_ACUTE,   DS_ACUTE,   _acute_eligible)

run_log_df = pd.DataFrame(RUN_LOG)
run_log_df.to_csv(REPORTS_DIR / "run_log.csv", index=False)
print(f"\nRun log → {REPORTS_DIR / 'run_log.csv'}")
display(run_log_df)


## 10 · Metrics — Dice / IoU / volume / surface distance

In [ ]:
def compute_metrics(ref_path: str, pred_path: str) -> dict:
    out = {"dice": np.nan, "iou": np.nan,
           "ref_vol_ml": np.nan, "pred_vol_ml": np.nan,
           "vol_diff_ml": np.nan, "vol_diff_pct": np.nan,
           "hd95_mm": np.nan, "mean_surf_mm": np.nan,
           "ref_voxels": np.nan, "pred_voxels": np.nan, "error": ""}
    if not ref_path or not pred_path:
        out["error"] = "missing path"; return out
    try:
        ref_img  = nib.load(ref_path)
        pred_img = nib.load(pred_path)
    except Exception as e:
        out["error"] = f"load: {type(e).__name__}: {e}"; return out
    try:
        pred_rs = resample_to_img(pred_img, ref_img, interpolation="nearest")
    except Exception as e:
        out["error"] = f"resample: {type(e).__name__}: {e}"; return out

    ref_arr  = binarise(ref_img); pred_arr = binarise(pred_rs)
    out["ref_voxels"]  = int(ref_arr.sum())
    out["pred_voxels"] = int(pred_arr.sum())
    out["dice"], out["iou"] = dice_iou(ref_arr, pred_arr)

    vv = voxel_volume_mm3(ref_img)
    out["ref_vol_ml"]   = volume_ml(ref_arr,  vv)
    out["pred_vol_ml"]  = volume_ml(pred_arr, vv)
    out["vol_diff_ml"]  = out["pred_vol_ml"] - out["ref_vol_ml"]
    out["vol_diff_pct"] = (100 * out["vol_diff_ml"] / out["ref_vol_ml"]
                           if out["ref_vol_ml"] else np.nan)
    if CONFIG["HD95"]:
        out.update(surface_distances(ref_arr, pred_arr,
                                     ref_img.header.get_zooms()[:3]))
    return out


metric_rows = []
for row in RUN_LOG:
    for tool in RUNNERS:
        pred = row.get(f"{tool}_pred", "")
        if not pred: continue
        m = compute_metrics(row["reference"], pred)
        m.update({"dataset": row["dataset"], "subject": row["subject"],
                  "session": row["session"], "tool": tool})
        metric_rows.append(m)

METRICS = pd.DataFrame(metric_rows)
if not METRICS.empty:
    cols = ["dataset","subject","session","tool","dice","iou",
            "vol_diff_ml","vol_diff_pct","hd95_mm","mean_surf_mm",
            "ref_vol_ml","pred_vol_ml","ref_voxels","pred_voxels","error"]
    METRICS = METRICS[[c for c in cols if c in METRICS.columns]]
    METRICS.to_csv(REPORTS_DIR / "per_subject_metrics.csv", index=False)
    print(f"Per-subject metrics → {REPORTS_DIR / 'per_subject_metrics.csv'}")
    display(METRICS.round(3))
else:
    print("No metrics — no (ref, pred) pair succeeded.")


## 11 · HarvardOxford per-region overlap

In [ ]:
from nilearn.datasets import fetch_atlas_harvard_oxford

def atlas_region_overlap(mask_path: str, atlas_img, atlas_labels) -> pd.DataFrame:
    if not mask_path:
        return pd.DataFrame()
    try:
        m_img = nib.load(mask_path)
    except Exception:
        return pd.DataFrame()
    m_rs = resample_to_img(m_img, atlas_img, interpolation="nearest")
    m    = binarise(m_rs)
    a    = atlas_img.get_fdata().astype(np.uint8)
    tot  = int(m.sum())
    if tot == 0:
        return pd.DataFrame()
    rows = []
    for idx, label in enumerate(atlas_labels):
        if idx == 0:  # background
            continue
        overlap = int(((a == idx) & (m == 1)).sum())
        if overlap >= 3:
            rows.append({"region": label, "overlap_voxels": overlap,
                         "overlap_pct": round(100 * overlap / tot, 2)})
    rows.sort(key=lambda r: -r["overlap_voxels"])
    return pd.DataFrame(rows)


print("Fetching HarvardOxford atlas (cached after first run) …")
ho = fetch_atlas_harvard_oxford(CONFIG["ATLAS"])
ATLAS_IMG, ATLAS_LABELS = ho.maps, ho.labels

region_rows = []
for row in RUN_LOG:
    sources = [("reference", row["reference"])] + [
        (tool, row.get(f"{tool}_pred", "")) for tool in RUNNERS
    ]
    for which, path in sources:
        df = atlas_region_overlap(path, ATLAS_IMG, ATLAS_LABELS)
        if df.empty: continue
        df["dataset"] = row["dataset"]
        df["subject"] = row["subject"]
        df["session"] = row["session"]
        df["source"]  = which
        region_rows.append(df)

REGIONS = pd.concat(region_rows, ignore_index=True) if region_rows else pd.DataFrame()
if not REGIONS.empty:
    REGIONS.to_csv(REPORTS_DIR / "region_overlap.csv", index=False)
    print(f"Region overlap → {REPORTS_DIR / 'region_overlap.csv'}  "
          f"({len(REGIONS)} rows)")
    display(REGIONS.head(20))
else:
    print("No region overlaps to report.")


## 12 · Aggregate report

In [ ]:
if METRICS.empty:
    display(HTML("<p style='color:#c00'>No metrics to aggregate.</p>"))
else:
    summary = (METRICS.groupby(["dataset", "tool"])
                       .agg(n=("dice", "count"),
                            dice_mean=("dice", "mean"),
                            dice_median=("dice", "median"),
                            iou_mean=("iou", "mean"),
                            vol_diff_ml_mean=("vol_diff_ml", "mean"),
                            vol_diff_pct_mean=("vol_diff_pct", "mean"),
                            hd95_mean=("hd95_mm", "mean"),
                            mean_surf_mean=("mean_surf_mm", "mean"))
                       .round(3).reset_index())
    summary.to_csv(REPORTS_DIR / "summary_by_tool.csv", index=False)
    print(f"Per-tool summary → {REPORTS_DIR / 'summary_by_tool.csv'}")
    display(summary)

    html = ['<div style="font-family:sans-serif;max-width:1100px">']
    html.append("<h2>Lesion-segmentation benchmark</h2>")
    html.append(f"<p style='color:#555'>Pilot: {CONFIG['N_CHRONIC']} chronic + "
                f"{CONFIG['N_ACUTE']} acute. Atlas: {CONFIG['ATLAS']}. "
                f"Predictions resampled (nearest-neighbour) to the "
                f"reference grid before comparison.</p>")
    html.append("<h3>Summary by dataset × tool</h3>")
    html.append(summary.to_html(index=False, classes="bench", border=0))
    for ds, sub in METRICS.groupby("dataset"):
        html.append(f"<h3>{ds}</h3>")
        html.append(sub.drop(columns=["error"]).round(3)
                       .to_html(index=False, classes="bench", border=0))
    html.append("<style>"
                ".bench { border-collapse:collapse; font-size:0.88em; margin:8px 0 }"
                ".bench th { background:#2c3e50; color:#fff; padding:5px 8px;"
                "            text-align:left }"
                ".bench td { padding:4px 8px; border-bottom:1px solid #eee }"
                "</style></div>")
    report_html = "\n".join(html)
    report_path = REPORTS_DIR / "benchmark_report.html"
    report_path.write_text(report_html)
    print(f"HTML report → {report_path}")
    display(HTML(report_html))


## Notes

- `module.load(...)` is async — Jupyter accepts top-level `await` thanks to IPython's autoawait. If running outside Neurodesk, the cell prints a warning and continues, and the runners just say "CLI not on PATH" when they hit that tool.
- `datalad install` is metadata-only (small). Actual NIfTI payload is fetched only for the subjects you sample.
- For chronic strokes the expert mask is in T2w-native space and LINDA's prediction is in T1w-native space. Resampling one to the other's grid is a rough proxy — for publication numbers you'd want ANTs to coregister T1↔T2 first.
- `CONFIG["SUBJECT_FILTER"] = ["sub-M2044", "sub-M2120"]` will force the benchmark onto specific subjects regardless of sampling.
